# RNN - Sentiment Analysis for IMDb data

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("IMDB Dataset.csv")

In [3]:
df.shape

(50000, 2)

In [4]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [6]:
df.drop_duplicates(inplace=True)

In [7]:
df.shape

(49582, 2)

## Pre-Processing of our data

__Steps to be performed:__
- Convert to Lowercase
- Remove URL, HTML Tags, Punctuations, Stop words(a, and, the, this...)
- Stemming
- Encode sentiment
- Vectorization (TF-IDF)

### 1. Convert to lowercase value

In [8]:
df["review"] = df["review"].str.lower()

### 2. Remove URL's

In [9]:
import re

sample_text = "abc is the word, abc" # abc => xyz

new_text = re.sub("abc", "xyz", sample_text)

In [10]:
new_text

'xyz is the word, xyz'

In [11]:
def remove_urls(text):
    text = re.sub( r"http\S+", "", text) # re.sub(Pattern, replacement, text)
    return text

df["review"] = df["review"].apply(remove_urls)

### 3. Remove Punctuations

In [12]:
def remove_punctuations(text):
    text = re.sub( r"[^A-Za-z0-9\s]", "", text) # A-Z a-z 0-9 \s(spaces) anything except this is a punctuation
    return text

In [13]:
df["review"] = df["review"].apply(remove_punctuations)

### 4. Remove HTML Tags

In [14]:
def remove_html(text):
    text = re.sub( r"<.*?>", "", text)
    return text

df["review"] = df["review"].apply(remove_html)

### 5. Removing Stopwords

In [15]:
import nltk # Natural Lang. ToolKit

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\aayus\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\aayus\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\aayus\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [16]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [17]:
sample_txt = "I like coding in python"
tokens = word_tokenize(sample_txt)

In [18]:
tokens

['I', 'like', 'coding', 'in', 'python']

In [19]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word, "")

    return text

df["review"] = df["review"].apply(remove_stopwords)

In [20]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


### 6. Stemming

In [21]:
# Get the words to the root/base form
# running -> run, playing -> play

# We'll use PorterStemming
from nltk.stem import PorterStemmer

In [22]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [23]:
df.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


### 7. Encoding

In [24]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [25]:
y = df["sentiment"]

In [26]:
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

### 8. Vectorization

In [27]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df["review"])

In [28]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 4057169 stored elements and shape (49582, 5000)>

## Datasets & Dataloaders

In [29]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [30]:
X_train.shape

(39665, 5000)

In [31]:
X_test.shape

(9917, 5000)

In [35]:
import torch
from torch.utils.data import TensorDataset, DataLoader

In [33]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [34]:
X_train

array([[0.2133337, 0.       , 0.       , ..., 0.       , 0.       ,
        0.       ],
       [0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        0.       ],
       [0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        0.       ],
       ...,
       [0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        0.       ],
       [0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        0.       ],
       [0.       , 0.       , 0.       , ..., 0.       , 0.       ,
        0.       ]], shape=(39665, 5000))

In [36]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float(),
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float(),
)

In [37]:
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)

## Build our RNN

In [46]:
import torch.nn as nn
import torch.optim as optim

In [47]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()
            
        self.hidden_size = hidden_size
        self.num_layers = num_layers
            
        # RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
            
        # fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # optional => shape (num of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        
        out, _= self.rnn(x, h0)
        # 1st value = hidden state of all the timesteps => (batch, seq_len, hidden size)
        # 2nd value = final hidden state of last timestep
        
        out = self.fc(out[:, -1, :])
        return out

In [48]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

## Training the RNN

In [50]:
# Unsqueeze, squeeze

epochs = 10
for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # Add singleton direction

        outputs = model(Xb) # (Batch_size, 1)
        
        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size, ) => probability
 
        loss = criterion(outputs, yb) # loss

        loss.backward() # BP
        optimizer.step() # Weights update

    print(f"Epoch: {epoch}, loss: {loss.item()}")

Epoch: 0, loss: 0.334723562002182
Epoch: 1, loss: 0.2636856436729431
Epoch: 2, loss: 0.2836828827857971
Epoch: 3, loss: 0.2416008561849594
Epoch: 4, loss: 0.18164198100566864
Epoch: 5, loss: 0.24941106140613556
Epoch: 6, loss: 0.18303921818733215
Epoch: 7, loss: 0.18002015352249146
Epoch: 8, loss: 0.18391908705234528
Epoch: 9, loss: 0.13544538617134094


## Evaluation

In [51]:
# Evaluate

model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0
    
    for Xb, yb in test_loader:
        Xb = Xb. unsqueeze(1)
    
        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()
    
        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

print(f"accuracy = {correct_vals/tot_vals*100}")

accuracy = 85.6609861853383
